# 06 — Week 2 Projections

This notebook converts the updated weekly team-strength ratings into game level projections for the selected target week.

The original preseason notebooks and preseason projections remain frozen.

For each target week matchup, this notebook:

- loads the updated weekly team strengths from Notebook 04
- preserves the original preseason Week 2 projection as a baseline
- recalculates expected margin using weekly strength, historical home field advantage, and rest
- converts expected margin into win probability using historical NFL margin variability
- carries forward the frozen preseason scoring baseline created for Week 1
- adds the new in season offensive and defensive scoring evidence
- produces projected scores, model spreads, ATS comparisons, totals comparisons, and final output tables
- saves all outputs only inside the weekly layer

The market is used only for comparison. It never enters the model projection itself.


In [28]:
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
from scipy.stats import norm
from sklearn.linear_model import LinearRegression


## Paths and Weekly Settings

For Week 2, `TARGET_WEEK = 2`.

Future weeks use the same notebook by changing the target week after the preceding weekly notebooks have been rerun.


In [29]:
PROJECT_ROOT = Path("../..")

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
WEEKLY_DATA_DIR = PROCESSED_DIR / "weekly"
WEEKLY_OUTPUT_DIR = PROJECT_ROOT / "weekly_projections" / "outputs"

SEASON = 2026
TARGET_WEEK = 12

WEEK_OUTPUT_DIR = (
    WEEKLY_OUTPUT_DIR
    / f"week_{TARGET_WEEK:02d}"
)

WEEK_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

WEEKLY_STRENGTH_PATH = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_sequential_team_strength.parquet"
)

INJURY_ADJUSTED_STRENGTH_PATH = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_injury_adjusted_team_strength.parquet"
)

TARGET_SCHEDULE_PATH = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_schedule.parquet"
)

PRESEASON_BASELINE_PATH = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_preseason_baseline.parquet"
)

WEEK1_PROJECTIONS_PATH = (
    WEEKLY_DATA_DIR
    / "week_01_projections.parquet"
)

# Load Weekly Inputs

Notebook 04 supplies the updated ratings. Notebook 02 supplies the target week schedule and frozen preseason comparison.


In [30]:
weekly_strength = pd.read_parquet(WEEKLY_STRENGTH_PATH)

injury_adjusted_strength = pd.read_parquet(
    INJURY_ADJUSTED_STRENGTH_PATH
)

target_schedule = pd.read_parquet(
    TARGET_SCHEDULE_PATH
)

preseason_baseline = pd.read_parquet(
    PRESEASON_BASELINE_PATH
)

target_schedule = target_schedule[
    target_schedule["home_score"].isna()
    & target_schedule["away_score"].isna()
].copy()


preseason_baseline = preseason_baseline[
    preseason_baseline["game_id"].isin(
        target_schedule["game_id"]
    )
].copy()


print(
    "Sequential weekly strengths:",
    weekly_strength.shape
)

print(
    "Injury-adjusted strengths:",
    injury_adjusted_strength.shape
)

print(
    "Remaining target-week games:",
    target_schedule.shape
)

print(
    "Frozen preseason games:",
    preseason_baseline.shape
)

assert len(weekly_strength) == 32
assert weekly_strength["team"].nunique() == 32

assert len(injury_adjusted_strength) == 32
assert injury_adjusted_strength["team"].nunique() == 32

assert len(target_schedule) == len(
    preseason_baseline
)

required_injury_columns = [
    "team",
    "weekly_team_strength",
    "injury_adjustment",
    "injury_adjusted_team_strength"
]

missing_injury_columns = [
    col
    for col in required_injury_columns
    if col not in injury_adjusted_strength.columns
]

assert not missing_injury_columns, (
    "Missing injury-adjusted columns: "
    + ", ".join(missing_injury_columns)
)


print(
    "Using sequential ratings as the pre-injury baseline "
    "and injury-adjusted ratings for Week "
    f"{TARGET_WEEK} projections."
)

Sequential weekly strengths: (32, 25)
Injury-adjusted strengths: (32, 30)
Remaining target-week games: (15, 46)
Frozen preseason games: (15, 16)
Using sequential ratings as the pre-injury baseline and injury-adjusted ratings for Week 12 projections.


### Injury adjusted game strength

For Week 2, the `home_weekly_team_strength` and `away_weekly_team_strength` values now include the conservative injury adjustment from Notebook 05.

The injury adjustment remains visible separately in the saved team strength file, so we can distinguish:

**preseason → performance update → injury update → final Week 2 rating**


# Attach Updated Team Strength

The matchup calculation is identical in structure to the original preseason game prediction model. The only change is the source of team strength:

`preseason team_strength` → `weekly_team_strength`

This isolates the in season update without changing the original model.


In [31]:
strength_lookup = injury_adjusted_strength[
    [
        "team",
        "weekly_team_strength",
        "injury_adjustment",
        "injury_adjusted_team_strength",
        "preseason_team_strength",
        "team_strength_change",
        "cumulative_offense_adjustment",
        "cumulative_defense_adjustment"
    ]
].copy()


games = target_schedule.copy()

home_lookup = strength_lookup.rename(
    columns={
        "team": "home_team",

        "weekly_team_strength":
            "home_pre_injury_team_strength",

        "injury_adjustment":
            "home_injury_adjustment",

        "injury_adjusted_team_strength":
            "home_weekly_team_strength",

        "preseason_team_strength":
            "home_preseason_team_strength",

        "team_strength_change":
            "home_strength_change",

        "cumulative_offense_adjustment":
            "home_offense_adjustment",

        "cumulative_defense_adjustment":
            "home_defense_adjustment"
    }
)

away_lookup = strength_lookup.rename(
    columns={
        "team": "away_team",

        "weekly_team_strength":
            "away_pre_injury_team_strength",

        "injury_adjustment":
            "away_injury_adjustment",

        "injury_adjusted_team_strength":
            "away_weekly_team_strength",

        "preseason_team_strength":
            "away_preseason_team_strength",

        "team_strength_change":
            "away_strength_change",

        "cumulative_offense_adjustment":
            "away_offense_adjustment",

        "cumulative_defense_adjustment":
            "away_defense_adjustment"
    }
)


games = (
    games
    .merge(
        home_lookup,
        on="home_team",
        how="left",
        validate="many_to_one"
    )
    .merge(
        away_lookup,
        on="away_team",
        how="left",
        validate="many_to_one"
    )
)

games["weekly_neutral_strength_diff"] = (
    games["home_weekly_team_strength"]
    - games["away_weekly_team_strength"]
)

games["pre_injury_neutral_strength_diff"] = (
    games["home_pre_injury_team_strength"]
    - games["away_pre_injury_team_strength"]
)


games["preseason_neutral_strength_diff"] = (
    games["home_preseason_team_strength"]
    - games["away_preseason_team_strength"]
)

assert games[
    "home_weekly_team_strength"
].notna().all()

assert games[
    "away_weekly_team_strength"
].notna().all()

assert games[
    "home_injury_adjustment"
].notna().all()

assert games[
    "away_injury_adjustment"
].notna().all()

In [32]:

# Verify injury adjustments entering Week 9 projections

audit = injury_adjusted_strength[
    [
        "team",
        "weekly_team_strength",
        "injury_adjustment",
        "injury_adjusted_team_strength"
    ]
].copy()

audit["calculated_strength"] = (
    audit["weekly_team_strength"]
    + audit["injury_adjustment"]
)

assert len(audit) == 32
assert audit["team"].nunique() == 32

assert np.allclose(
    audit["calculated_strength"],
    audit["injury_adjusted_team_strength"]
)

print("Projection week:", TARGET_WEEK)
print("Teams validated:", len(audit))
print(
    "Teams with active injury adjustments:",
    int((audit["injury_adjustment"] != 0).sum())
)
print(
    "Total applied injury adjustment:",
    round(audit["injury_adjustment"].sum(), 3)
)

display(
    audit.sort_values("injury_adjustment").round(3)
)

if (audit["injury_adjustment"] == 0).all():
    print(
        "WARNING: No verified injury penalties are currently "
        "applied. Projections will be preliminary."
    )


Projection week: 12
Teams validated: 32
Teams with active injury adjustments: 32
Total applied injury adjustment: -40.407


,team,weekly_team_strength,injury_adjustment,injury_adjusted_team_strength,calculated_strength
19,BAL,-2.549,-4.500,-7.049,-7.049
30,ARI,-7.289,-4.422,-11.711,-11.711
23,PHI,-3.439,-3.118,-6.556,-6.556
28,MIA,-5.107,-3.035,-8.141,-8.141
4,DET,5.463,-1.631,3.832,3.832
26,LAC,-4.734,-1.611,-6.344,-6.344
1,SF,6.573,-1.587,4.987,4.987
3,LA,5.982,-1.381,4.601,4.601
16,NE,0.126,-1.280,-1.154,-1.154
15,DAL,0.422,-1.275,-0.854,-0.854


# Home Field and Rest

These effects are recalculated using the same historical 2015–2025 methodology as the frozen preseason game prediction notebook.

They are not retuned using Week 1.


In [33]:
schedule_clean = pl.read_parquet(
    PROCESSED_DIR / "schedule_clean.parquet"
)

historical_home = (
    schedule_clean
    .filter(
        (pl.col("season") >= 2015)
        & (pl.col("season") <= 2025)
        & (pl.col("location") == "Home")
        & pl.col("result").is_not_null()
    )
)

HOME_FIELD_ADVANTAGE = (
    historical_home
    .select(pl.col("result").mean())
    .item()
)

historical_rest = (
    schedule_clean
    .filter(
        (pl.col("season") >= 2015)
        & (pl.col("season") <= 2025)
        & pl.col("result").is_not_null()
        & pl.col("home_rest").is_not_null()
        & pl.col("away_rest").is_not_null()
    )
    .select(["result", "home_rest", "away_rest", "location"])
    .to_pandas()
)

historical_rest["rest_diff"] = (
    historical_rest["home_rest"]
    - historical_rest["away_rest"]
).clip(-7, 7)

historical_rest["home_field"] = (
    historical_rest["location"] == "Home"
).astype(int)

rest_model = LinearRegression()
rest_model.fit(
    historical_rest[["home_field", "rest_diff"]],
    historical_rest["result"]
)

REST_POINT_VALUE = rest_model.coef_[1]

print("Historical HFA:", round(HOME_FIELD_ADVANTAGE, 3))
print("Rest points/day:", round(REST_POINT_VALUE, 3))


Historical HFA: 1.764
Rest points/day: 0.181


In [34]:
games["home_field_adjustment"] = np.where(
    games["location"] == "Home",
    HOME_FIELD_ADVANTAGE,
    0.0
)

games["rest_diff"] = (
    games["home_rest"]
    - games["away_rest"]
).clip(-7, 7)

games["rest_adjustment"] = (
    games["rest_diff"]
    * REST_POINT_VALUE
)

games["expected_home_margin"] = (
    games["weekly_neutral_strength_diff"]
    + games["home_field_adjustment"]
    + games["rest_adjustment"]
)


# Win Probabilities

Historical 2015–2025 game-margin variability is retained exactly as the uncertainty scale. Week 1 is not used to recalibrate probability confidence.


In [35]:
historical_margin_std = (
    schedule_clean
    .filter(
        (pl.col("season") >= 2015)
        & (pl.col("season") <= 2025)
        & pl.col("result").is_not_null()
    )
    .select(pl.col("result").std())
    .item()
)

games["home_win_probability"] = norm.cdf(
    games["expected_home_margin"]
    / historical_margin_std
)

games["away_win_probability"] = (
    1 - games["home_win_probability"]
)

games["predicted_winner"] = np.where(
    games["home_win_probability"] >= 0.50,
    games["home_team"],
    games["away_team"]
)

games["predicted_win_probability"] = np.maximum(
    games["home_win_probability"],
    games["away_win_probability"]
)

print("Historical margin SD:", round(historical_margin_std, 3))


Historical margin SD: 14.201


# Preseason vs Weekly Margin Movement

The frozen preseason Week 2 projection is attached for diagnostics only.

This makes it possible to see exactly how much the Week 1 evidence changed each Week 2 matchup.


In [36]:
preseason_compare = preseason_baseline[
    [
        "game_id",
        "expected_home_margin",
        "predicted_winner",
        "predicted_win_probability"
    ]
].rename(columns={
    "expected_home_margin": "preseason_expected_home_margin",
    "predicted_winner": "preseason_predicted_winner",
    "predicted_win_probability": "preseason_predicted_win_probability"
})

games = games.merge(
    preseason_compare,
    on="game_id",
    how="left",
    validate="one_to_one"
)

games["margin_change_from_preseason"] = (
    games["expected_home_margin"]
    - games["preseason_expected_home_margin"]
)

display(
    games[
        [
            "away_team",
            "home_team",
            "preseason_expected_home_margin",
            "expected_home_margin",
            "margin_change_from_preseason",
            "preseason_predicted_winner",
            "predicted_winner",
            "predicted_win_probability"
        ]
    ].round(3)
)


,away_team,home_team,preseason_expected_home_margin,expected_home_margin,margin_change_from_preseason,preseason_predicted_winner,predicted_winner,predicted_win_probability
0,CHI,DET,5.834,3.429,-2.405,DET,DET,0.595
1,KC,BUF,5.278,4.501,-0.776,BUF,BUF,0.624
2,PHI,DAL,-1.075,7.466,8.541,PHI,DAL,0.700
3,DEN,PIT,-1.262,-2.227,-0.965,DEN,DEN,0.562
4,ATL,MIN,2.303,6.194,3.891,MIN,MIN,0.669
5,BAL,HOU,2.043,11.344,9.302,HOU,HOU,0.788
6,LV,CLE,4.573,4.314,-0.259,CLE,CLE,0.619
7,NE,LAC,-0.503,-4.694,-4.191,NE,NE,0.630
8,NO,CIN,2.399,8.707,6.307,CIN,CIN,0.730
9,NYG,IND,6.295,-1.305,-7.599,IND,NYG,0.537


# Frozen Preseason Scoring Ratings

Week 1's saved projection artifact contains the preseason offensive/defensive scoring ratings and personnel scoring adjustments for all 32 teams.

Those values were created before the season and are reused here as the frozen scoring prior. We do not rebuild or modify the preseason notebooks.

The new weekly offense/defense signals are added separately afterward.


In [37]:
week1_frozen = pd.read_parquet(
    WEEK1_PROJECTIONS_PATH
)

home_scoring = week1_frozen[
    [
        "home_team",
        "home_projected_off_ppg",
        "home_projected_def_ppg",
        "home_off_personnel_z",
        "home_def_personnel_z"
    ]
].rename(columns={
    "home_team": "team",
    "home_projected_off_ppg": "preseason_projected_off_ppg",
    "home_projected_def_ppg": "preseason_projected_def_ppg",
    "home_off_personnel_z": "off_personnel_z",
    "home_def_personnel_z": "def_personnel_z"
})

away_scoring = week1_frozen[
    [
        "away_team",
        "away_projected_off_ppg",
        "away_projected_def_ppg",
        "away_off_personnel_z",
        "away_def_personnel_z"
    ]
].rename(columns={
    "away_team": "team",
    "away_projected_off_ppg": "preseason_projected_off_ppg",
    "away_projected_def_ppg": "preseason_projected_def_ppg",
    "away_off_personnel_z": "off_personnel_z",
    "away_def_personnel_z": "def_personnel_z"
})

scoring_prior = (
    pd.concat([home_scoring, away_scoring], ignore_index=True)
    .groupby("team", as_index=False)
    .agg(
        preseason_projected_off_ppg=("preseason_projected_off_ppg", "mean"),
        preseason_projected_def_ppg=("preseason_projected_def_ppg", "mean"),
        off_personnel_z=("off_personnel_z", "mean"),
        def_personnel_z=("def_personnel_z", "mean")
    )
)

assert len(scoring_prior) == 32
display(scoring_prior.head())


,team,preseason_projected_off_ppg,preseason_projected_def_ppg,off_personnel_z,def_personnel_z
0,ARI,22.191473,24.425844,-0.763765,-1.345326
1,ATL,22.032425,23.261103,0.507936,0.513715
2,BAL,25.007820,22.580763,0.442967,0.969729
3,BUF,25.942602,22.289034,1.060099,0.231615
4,CAR,20.533474,23.731353,-1.506345,-0.346889


In [38]:
completed_games_path = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_completed_games.parquet"
)

completed_games = pd.read_parquet(
    completed_games_path
).copy()

completed_games = completed_games[
    completed_games["week"] < TARGET_WEEK
].copy()

home_scoring_actual = completed_games[
    [
        "home_team",
        "home_score",
        "away_score"
    ]
].rename(
    columns={
        "home_team": "team",
        "home_score": "points_for",
        "away_score": "points_against"
    }
)

away_scoring_actual = completed_games[
    [
        "away_team",
        "away_score",
        "home_score"
    ]
].rename(
    columns={
        "away_team": "team",
        "away_score": "points_for",
        "home_score": "points_against"
    }
)

team_scoring_actual = pd.concat(
    [
        home_scoring_actual,
        away_scoring_actual
    ],
    ignore_index=True
)

inseason_scoring = (
    team_scoring_actual
    .groupby("team", as_index=False)
    .agg(
        games_played=("points_for", "size"),
        actual_off_ppg=("points_for", "mean"),
        actual_def_ppg=("points_against", "mean")
    )
)

assert len(inseason_scoring) == 32

assert (
    inseason_scoring["games_played"] > 0
).all(), "Some teams have no completed games."

assert (
    inseason_scoring["games_played"] <= TARGET_WEEK - 1
).all(), "A team has more completed games than expected."

print("Games played distribution:")
print(
    inseason_scoring["games_played"]
    .value_counts()
    .sort_index()
)

print("\nTeams with fewer completed games:")
display(
    inseason_scoring.loc[
        inseason_scoring["games_played"] < TARGET_WEEK - 1,
        ["team", "games_played"]
    ].sort_values("games_played")
)

display(
    inseason_scoring
    .sort_values("actual_off_ppg", ascending=False)
    .round(2)
)

Games played distribution:
games_played
10    26
11     6
Name: count, dtype: int64

Teams with fewer completed games:


,team,games_played
1,ATL,10
3,BUF,10
4,CAR,10
5,CHI,10
6,CIN,10
7,CLE,10
9,DEN,10
10,DET,10
11,GB,10
12,HOU,10


,team,games_played,actual_off_ppg,actual_def_ppg
28,SF,10,31.60,20.60
3,BUF,10,31.30,20.80
8,DAL,11,30.64,28.36
6,CIN,10,29.80,21.50
10,DET,10,28.80,22.00
14,JAX,10,28.70,16.70
15,KC,10,27.10,16.30
5,CHI,10,26.20,19.70
12,HOU,10,25.90,22.20
4,CAR,10,25.40,26.90


In [39]:
scoring_expectations = scoring_prior[
    [
        "team",
        "preseason_projected_off_ppg",
        "preseason_projected_def_ppg"
    ]
].copy()

home_games = completed_games[
    [
        "week",
        "home_team",
        "away_team",
        "home_score",
        "away_score"
    ]
].copy()

home_games = home_games.rename(
    columns={
        "home_team": "team",
        "away_team": "opponent",
        "home_score": "points_for",
        "away_score": "points_against"
    }
)

away_games = completed_games[
    [
        "week",
        "away_team",
        "home_team",
        "away_score",
        "home_score"
    ]
].copy()

away_games = away_games.rename(
    columns={
        "away_team": "team",
        "home_team": "opponent",
        "away_score": "points_for",
        "home_score": "points_against"
    }
)

team_games = pd.concat(
    [home_games, away_games],
    ignore_index=True
)

opponent_defense = scoring_expectations[
    [
        "team",
        "preseason_projected_def_ppg"
    ]
].rename(
    columns={
        "team": "opponent",
        "preseason_projected_def_ppg":
            "opponent_expected_def_ppg"
    }
)

opponent_offense = scoring_expectations[
    [
        "team",
        "preseason_projected_off_ppg"
    ]
].rename(
    columns={
        "team": "opponent",
        "preseason_projected_off_ppg":
            "opponent_expected_off_ppg"
    }
)

team_games = (
    team_games
    .merge(
        opponent_defense,
        on="opponent",
        how="left",
        validate="many_to_one"
    )
    .merge(
        opponent_offense,
        on="opponent",
        how="left",
        validate="many_to_one"
    )
)

league_preseason_off_ppg = (
    scoring_prior[
        "preseason_projected_off_ppg"
    ].mean()
)

league_preseason_def_ppg = (
    scoring_prior[
        "preseason_projected_def_ppg"
    ].mean()
)

team_games["adjusted_points_for"] = (
    team_games["points_for"]
    + (
        league_preseason_def_ppg
        - team_games["opponent_expected_def_ppg"]
    )
)

team_games["adjusted_points_against"] = (
    team_games["points_against"]
    + (
        league_preseason_off_ppg
        - team_games["opponent_expected_off_ppg"]
    )
)

inseason_scoring_adjusted = (
    team_games
    .groupby("team", as_index=False)
    .agg(
        games_played=("points_for", "size"),
        actual_off_ppg=("points_for", "mean"),
        actual_def_ppg=("points_against", "mean"),
        adjusted_off_ppg=(
            "adjusted_points_for",
            "mean"
        ),
        adjusted_def_ppg=(
            "adjusted_points_against",
            "mean"
        )
    )
)

assert len(inseason_scoring_adjusted) == 32

display(
    inseason_scoring_adjusted[
        [
            "team",
            "actual_off_ppg",
            "adjusted_off_ppg",
            "actual_def_ppg",
            "adjusted_def_ppg"
        ]
    ]
    .sort_values(
        "adjusted_off_ppg",
        ascending=False
    )
    .round(2)
)

,team,actual_off_ppg,adjusted_off_ppg,actual_def_ppg,adjusted_def_ppg
3,BUF,31.30,31.57,20.80,20.74
28,SF,31.60,31.44,20.60,20.65
8,DAL,30.64,30.46,28.36,28.23
6,CIN,29.80,29.76,21.50,21.62
14,JAX,28.70,28.74,16.70,16.82
10,DET,28.80,28.73,22.00,22.41
15,KC,27.10,27.11,16.30,16.80
5,CHI,26.20,26.46,19.70,20.16
4,CAR,25.40,25.70,26.90,26.68
18,LV,25.00,25.52,29.73,29.44


In [40]:
scoring_model = scoring_prior.merge(
    inseason_scoring_adjusted,
    on="team",
    how="left",
    validate="one_to_one"
)

PRESEASON_EQUIVALENT_GAMES = 5.0

scoring_model["preseason_weight"] = (
    PRESEASON_EQUIVALENT_GAMES
    / (
        PRESEASON_EQUIVALENT_GAMES
        + scoring_model["games_played"]
    )
)

scoring_model["inseason_weight"] = (
    1.0
    - scoring_model["preseason_weight"]
)

scoring_model["current_off_ppg"] = (
    scoring_model["preseason_weight"]
    * scoring_model["preseason_projected_off_ppg"]
    +
    scoring_model["inseason_weight"]
    * scoring_model["adjusted_off_ppg"]
)

scoring_model["current_def_ppg"] = (
    scoring_model["preseason_weight"]
    * scoring_model["preseason_projected_def_ppg"]
    +
    scoring_model["inseason_weight"]
    * scoring_model["adjusted_def_ppg"]
)

assert len(scoring_model) == 32

assert scoring_model[
    [
        "current_off_ppg",
        "current_def_ppg"
    ]
].notna().all().all()

display(
    scoring_model[
        [
            "team",
            "preseason_projected_off_ppg",
            "adjusted_off_ppg",
            "current_off_ppg",
            "preseason_projected_def_ppg",
            "adjusted_def_ppg",
            "current_def_ppg",
            "preseason_weight",
            "inseason_weight"
        ]
    ]
    .sort_values(
        "current_off_ppg",
        ascending=False
    )
    .round(2)
)

,team,preseason_projected_off_ppg,adjusted_off_ppg,current_off_ppg,preseason_projected_def_ppg,adjusted_def_ppg,current_def_ppg,preseason_weight,inseason_weight
3,BUF,25.94,31.57,29.69,22.29,20.74,21.26,0.33,0.67
28,SF,24.31,31.44,29.07,22.70,20.65,21.33,0.33,0.67
8,DAL,24.71,30.46,28.66,24.75,28.23,27.14,0.31,0.69
10,DET,26.28,28.73,27.91,23.05,22.41,22.62,0.33,0.67
6,CIN,23.96,29.76,27.82,24.55,21.62,22.59,0.33,0.67
14,JAX,23.95,28.74,27.14,22.47,16.82,18.71,0.33,0.67
15,KC,22.39,27.11,25.54,21.52,16.80,18.37,0.33,0.67
5,CHI,23.20,26.46,25.37,23.18,20.16,21.16,0.33,0.67
16,LA,25.21,24.67,24.85,22.35,18.87,20.03,0.33,0.67
12,HOU,23.08,25.41,24.63,21.53,22.01,21.85,0.33,0.67


# Weekly Scoring Update

The in-season offensive and defensive adjustments from Notebook 04 are already shrunk for sample size.

They are added to the frozen scoring prior:

- positive offensive adjustment raises expected scoring
- positive defensive adjustment represents points saved, so it lowers projected points allowed

This keeps the Week 1 scoring evidence separate and interpretable.


In [41]:
scoring_update = scoring_model[
    [
        "team",
        "current_off_ppg",
        "current_def_ppg"
    ]
].copy()

assert len(scoring_update) == 32
assert scoring_update["team"].nunique() == 32

display(
    scoring_update
    .sort_values(
        "current_off_ppg",
        ascending=False
    )
    .round(2)
)

,team,current_off_ppg,current_def_ppg
3,BUF,29.69,21.26
28,SF,29.07,21.33
8,DAL,28.66,27.14
10,DET,27.91,22.62
6,CIN,27.82,22.59
14,JAX,27.14,18.71
15,KC,25.54,18.37
5,CHI,25.37,21.16
16,LA,24.85,20.03
12,HOU,24.63,21.85


# Weekly Projected Total

The matchup total uses the same basic preseason scoring concept: each offense is paired with the opposing defense.

Because the weekly offense/defense inputs are already heavily shrunk, no additional Week 1 multiplier is applied here.

The final projected scores are forced to satisfy both:

- projected score difference = weekly expected margin
- projected score sum = weekly projected total


In [42]:
home_scoring_update = scoring_update.rename(
    columns={
        "team": "home_team",
        "current_off_ppg":
            "home_current_off_ppg",
        "current_def_ppg":
            "home_current_def_ppg"
    }
)

away_scoring_update = scoring_update.rename(
    columns={
        "team": "away_team",
        "current_off_ppg":
            "away_current_off_ppg",
        "current_def_ppg":
            "away_current_def_ppg"
    }
)

In [43]:
games = (
    games
    .merge(
        home_scoring_update,
        on="home_team",
        how="left",
        validate="many_to_one"
    )
    .merge(
        away_scoring_update,
        on="away_team",
        how="left",
        validate="many_to_one"
    )
)

required_scoring_columns = [
    "home_current_off_ppg",
    "home_current_def_ppg",
    "away_current_off_ppg",
    "away_current_def_ppg"
]

assert games[
    required_scoring_columns
].notna().all().all()

games["raw_home_score"] = (
    games["home_current_off_ppg"]
    + games["away_current_def_ppg"]
) / 2

games["raw_away_score"] = (
    games["away_current_off_ppg"]
    + games["home_current_def_ppg"]
) / 2

games["projected_total"] = (
    games["raw_home_score"]
    + games["raw_away_score"]
)

games["projected_home_score"] = (
    games["projected_total"]
    + games["expected_home_margin"]
) / 2

games["projected_away_score"] = (
    games["projected_total"]
    - games["expected_home_margin"]
) / 2


assert games["projected_total"].notna().all()

assert np.allclose(
    games["projected_home_score"]
    - games["projected_away_score"],
    games["expected_home_margin"]
)

assert np.allclose(
    games["projected_home_score"]
    + games["projected_away_score"],
    games["projected_total"]
)


display(
    games[
        [
            "away_team",
            "home_team",
            "away_current_off_ppg",
            "home_current_def_ppg",
            "home_current_off_ppg",
            "away_current_def_ppg",
            "projected_away_score",
            "projected_home_score",
            "projected_total",
            "expected_home_margin"
        ]
    ].round(2)
)

,away_team,home_team,away_current_off_ppg,home_current_def_ppg,home_current_off_ppg,away_current_def_ppg,projected_away_score,projected_home_score,projected_total,expected_home_margin
0,CHI,DET,25.37,22.62,27.91,21.16,22.55,25.98,48.54,3.43
1,KC,BUF,25.54,21.26,29.69,18.37,21.46,25.97,47.43,4.50
2,PHI,DAL,18.57,27.14,28.66,23.19,20.66,28.13,48.78,7.47
3,DEN,PIT,23.75,21.13,21.68,21.34,23.09,20.86,43.95,-2.23
4,ATL,MIN,20.79,17.43,19.91,24.90,17.66,23.86,41.52,6.19
5,BAL,HOU,22.67,21.85,24.63,25.62,18.02,29.37,47.39,11.34
6,LV,CLE,23.63,22.75,19.18,27.61,21.14,25.45,46.59,4.31
7,NE,LAC,22.57,24.89,18.93,23.32,24.78,20.08,44.86,-4.69
8,NO,CIN,21.31,22.59,27.82,25.15,19.87,28.57,48.44,8.71
9,NYG,IND,21.97,25.92,20.89,22.19,23.39,22.09,45.49,-1.30


# Market Comparison

nflverse uses positive `spread_line` for a home favorite and negative for an away favorite. The values below are converted to standard sportsbook notation.

Market information is used only after the model projections have been completed.


In [44]:
games["home_spread"] = -games["spread_line"]
games["away_spread"] = games["spread_line"]

games["model_home_spread"] = -games["expected_home_margin"]
games["model_away_spread"] = games["expected_home_margin"]

games["home_ats_edge"] = (
    games["home_spread"]
    - games["model_home_spread"]
)

games["away_ats_edge"] = (
    games["away_spread"]
    - games["model_away_spread"]
)

games["ats_pick"] = np.where(
    games["home_ats_edge"] > games["away_ats_edge"],
    games["home_team"],
    np.where(
        games["away_ats_edge"] > games["home_ats_edge"],
        games["away_team"],
        "PUSH"
    )
)

games["ats_difference"] = (
    games[["home_ats_edge", "away_ats_edge"]]
    .max(axis=1)
)

games["total_difference_signed"] = (
    games["projected_total"]
    - games["total_line"]
)

games["total_difference"] = (
    games["total_difference_signed"].abs()
)

games["total_pick"] = np.where(
    games["total_difference_signed"] > 0,
    "OVER",
    np.where(
        games["total_difference_signed"] < 0,
        "UNDER",
        "PUSH"
    )
)


## Unit Sizing

Unit sizes are assigned from the model's edge over the market.

ATS and totals use separate thresholds because the totals model is newer and requires a larger edge before qualifying as a play.

These units do not change any projections or picks. They provide a consistent way to track performance by model confidence.

In [45]:
def assign_ats_units(edge):
    if pd.isna(edge) or edge < 1.0:
        return 0.0
    elif edge < 2.0:
        return 0.5
    elif edge < 3.0:
        return 1.0
    elif edge < 4.0:
        return 1.5
    else:
        return 2.0


def assign_total_units(edge):
    if pd.isna(edge) or edge < 2.0:
        return 0.0
    elif edge < 3.5:
        return 0.5
    elif edge < 5.0:
        return 1.0
    elif edge < 7.0:
        return 1.5
    else:
        return 2.0


games["ats_units"] = games["ats_difference"].apply(assign_ats_units)

games["total_units"] = games["total_difference"].apply(assign_total_units)


games[
    [
        "away_team",
        "home_team",
        "ats_pick",
        "ats_difference",
        "ats_units",
        "total_pick",
        "total_difference",
        "total_units"
    ]
]

,away_team,home_team,ats_pick,ats_difference,ats_units,total_pick,total_difference,total_units
0,CHI,DET,DET,4.928737,2.0,UNDER,7.963724,2.0
1,KC,BUF,BUF,1.501366,0.5,OVER,1.930365,0.0
2,PHI,DAL,PHI,1.033979,0.5,UNDER,0.715662,0.0
3,DEN,PIT,PIT,2.273228,1.0,OVER,7.450532,2.0
4,ATL,MIN,ATL,0.805655,0.0,OVER,0.018734,0.0
5,BAL,HOU,HOU,4.844102,2.0,UNDER,0.110972,0.0
6,LV,CLE,LV,0.185872,0.0,OVER,6.088737,1.5
7,NE,LAC,NE,2.194212,1.0,OVER,9.360229,2.0
8,NO,CIN,NO,1.293455,0.5,UNDER,1.062121,0.0
9,NYG,IND,IND,4.695170,2.0,OVER,3.985068,1.0


# Display Helpers


In [46]:
def format_spread(team, line):
    if pd.isna(line):
        return "N/A"
    if line > 0:
        return f"{team} +{line:.1f}"
    return f"{team} {line:.1f}"

def model_spread_display(row):
    if row["model_home_spread"] < 0:
        return format_spread(row["home_team"], row["model_home_spread"])
    if row["model_away_spread"] < 0:
        return format_spread(row["away_team"], row["model_away_spread"])
    return "PICK"

def market_spread_display(row):
    if pd.isna(row["spread_line"]):
        return "N/A"
    if row["home_spread"] < 0:
        return format_spread(row["home_team"], row["home_spread"])
    if row["away_spread"] < 0:
        return format_spread(row["away_team"], row["away_spread"])
    return "PICK"

def ats_pick_display(row):
    if row["ats_pick"] == "PUSH":
        return "PUSH"
    if row["ats_pick"] == row["home_team"]:
        return format_spread(row["home_team"], row["home_spread"])
    return format_spread(row["away_team"], row["away_spread"])

games["projected_score"] = (
    games["away_team"]
    + " "
    + games["projected_away_score"].round(1).astype(str)
    + " - "
    + games["home_team"]
    + " "
    + games["projected_home_score"].round(1).astype(str)
)

games["model_spread_display"] = games.apply(
    model_spread_display, axis=1
)

games["market_spread_display"] = games.apply(
    market_spread_display, axis=1
)

games["ats_pick_display"] = games.apply(
    ats_pick_display, axis=1
)


# Final Week Projection Board


In [47]:

projection_board = pd.DataFrame({
    "Matchup": games["away_team"] + " @ " + games["home_team"],
    "Projected Score": games["projected_score"],
    "SU Pick": games["predicted_winner"],
    "Win Prob": games["predicted_win_probability"],
    "Model Spread": games["model_spread_display"],
    "Market Spread": games["market_spread_display"],
    "ATS Pick": games["ats_pick_display"],
    "ATS Diff": games["ats_difference"],
    "ATS Units": games["ats_units"].map(lambda x: f"{x:.1f}"),
    "Model Total": games["projected_total"],
    "Market Total": games["total_line"],
    "O/U Pick": games["total_pick"],
    "Total Diff": games["total_difference"],
    "Total Units": games["total_units"].map(lambda x: f"{x:.1f}"),
    "Preseason Margin": games["preseason_expected_home_margin"],
    "Weekly Margin": games["expected_home_margin"],
    "Margin Change": games["margin_change_from_preseason"]
})

projection_display = projection_board.copy()

projection_csv_path = (
    WEEK_OUTPUT_DIR
    / f"week_{TARGET_WEEK:02d}_projections.csv"
)

if projection_csv_path.exists():
    previous_board = pd.read_csv(projection_csv_path)

    current_matchups = set(projection_display["Matchup"])

    frozen_board = previous_board.loc[
        ~previous_board["Matchup"].isin(current_matchups)
    ].copy()

    projection_display = pd.concat(
        [frozen_board, projection_display],
        ignore_index=True
    )

assert projection_display["Matchup"].is_unique, (
    "Duplicate matchups found in projection board."
)

display_board = projection_display.copy()

numeric_columns = [
    "Win Prob",
    "ATS Diff",
    "ATS Units",
    "Model Total",
    "Market Total",
    "Total Diff",
    "Total Units",
    "Preseason Margin",
    "Weekly Margin",
    "Margin Change"
]

for col in numeric_columns:
    display_board[col] = pd.to_numeric(
        display_board[col],
        errors="raise"
    )

print(
    f"Week {TARGET_WEEK} Projection Board "
    f"({len(display_board)} games)"
)

display(
    display_board.style.format({
        "Win Prob": "{:.3f}",
        "ATS Diff": "{:.2f}",
        "ATS Units": "{:.1f}",
        "Model Total": "{:.1f}",
        "Market Total": "{:.1f}",
        "Total Diff": "{:.1f}",
        "Total Units": "{:.1f}",
        "Preseason Margin": "{:.2f}",
        "Weekly Margin": "{:.2f}",
        "Margin Change": "{:+.2f}"
    }).hide(axis="index")
)


Week 12 Projection Board (16 games)


Matchup,Projected Score,SU Pick,Win Prob,Model Spread,Market Spread,ATS Pick,ATS Diff,ATS Units,Model Total,Market Total,O/U Pick,Total Diff,Total Units,Preseason Margin,Weekly Margin,Margin Change
GB @ LA,GB 19.7 - LA 25.2,LA,0.649,LA -5.4,LA -2.5,LA -2.5,2.94,1.0,44.9,43.5,OVER,1.4,0.0,3.99,5.44,+1.45
CHI @ DET,CHI 22.6 - DET 26.0,DET,0.595,DET -3.4,CHI -1.5,DET +1.5,4.93,2.0,48.5,56.5,UNDER,8.0,2.0,5.83,3.43,-2.41
KC @ BUF,KC 21.5 - BUF 26.0,BUF,0.624,BUF -4.5,BUF -3.0,BUF -3.0,1.50,0.5,47.4,45.5,OVER,1.9,0.0,5.28,4.50,-0.78
PHI @ DAL,PHI 20.7 - DAL 28.1,DAL,0.700,DAL -7.5,DAL -8.5,PHI +8.5,1.03,0.5,48.8,49.5,UNDER,0.7,0.0,-1.07,7.47,+8.54
DEN @ PIT,DEN 23.1 - PIT 20.9,DEN,0.562,DEN -2.2,DEN -4.5,PIT +4.5,2.27,1.0,44.0,36.5,OVER,7.5,2.0,-1.26,-2.23,-0.96
ATL @ MIN,ATL 17.7 - MIN 23.9,MIN,0.669,MIN -6.2,MIN -7.0,ATL +7.0,0.81,0.0,41.5,41.5,OVER,0.0,0.0,2.30,6.19,+3.89
BAL @ HOU,BAL 18.0 - HOU 29.4,HOU,0.788,HOU -11.3,HOU -6.5,HOU -6.5,4.84,2.0,47.4,47.5,UNDER,0.1,0.0,2.04,11.34,+9.30
LV @ CLE,LV 21.1 - CLE 25.5,CLE,0.619,CLE -4.3,CLE -4.5,LV +4.5,0.19,0.0,46.6,40.5,OVER,6.1,1.5,4.57,4.31,-0.26
NE @ LAC,NE 24.8 - LAC 20.1,NE,0.630,NE -4.7,NE -2.5,NE -2.5,2.19,1.0,44.9,35.5,OVER,9.4,2.0,-0.50,-4.69,-4.19
NO @ CIN,NO 19.9 - CIN 28.6,CIN,0.730,CIN -8.7,CIN -10.0,NO +10.0,1.29,0.5,48.4,49.5,UNDER,1.1,0.0,2.40,8.71,+6.31


# Biggest Model vs Market Differences


In [48]:
biggest_ats_differences = (
    projection_board[
        [
            "Matchup",
            "Projected Score",
            "Model Spread",
            "Market Spread",
            "ATS Pick",
            "ATS Diff"
        ]
    ]
    .sort_values("ATS Diff", ascending=False)
    .head(5)
    .copy()
)

biggest_total_differences = (
    projection_board[
        [
            "Matchup",
            "Projected Score",
            "Model Total",
            "Market Total",
            "O/U Pick",
            "Total Diff"
        ]
    ]
    .sort_values("Total Diff", ascending=False)
    .head(5)
    .copy()
)

print("BIGGEST ATS DIFFERENCES")
display(biggest_ats_differences.round(2))

print()
print("BIGGEST TOTAL DIFFERENCES")
display(biggest_total_differences.round(2))


BIGGEST ATS DIFFERENCES


,Matchup,Projected Score,Model Spread,Market Spread,ATS Pick,ATS Diff
10,NYJ @ MIA,NYJ 21.6 - MIA 22.2,MIA -0.6,NYJ -4.5,MIA +4.5,5.08
0,CHI @ DET,CHI 22.6 - DET 26.0,DET -3.4,CHI -1.5,DET +1.5,4.93
5,BAL @ HOU,BAL 18.0 - HOU 29.4,HOU -11.3,HOU -6.5,HOU -6.5,4.84
9,NYG @ IND,NYG 23.4 - IND 22.1,NYG -1.3,NYG -6.0,IND +6.0,4.70
14,CAR @ TB,CAR 20.5 - TB 26.4,TB -6.0,TB -3.0,TB -3.0,2.98



BIGGEST TOTAL DIFFERENCES


,Matchup,Projected Score,Model Total,Market Total,O/U Pick,Total Diff
7,NE @ LAC,NE 24.8 - LAC 20.1,44.86,35.5,OVER,9.36
0,CHI @ DET,CHI 22.6 - DET 26.0,48.54,56.5,UNDER,7.96
3,DEN @ PIT,DEN 23.1 - PIT 20.9,43.95,36.5,OVER,7.45
6,LV @ CLE,LV 21.1 - CLE 25.5,46.59,40.5,OVER,6.09
13,WAS @ ARI,WAS 27.0 - ARI 19.1,46.05,41.5,OVER,4.55


In [49]:
biggest_ats_differences["ATS Diff"] = (
    biggest_ats_differences["ATS Diff"].round(2)
)

biggest_total_differences["Model Total"] = (
    biggest_total_differences["Model Total"].round(1)
)

biggest_total_differences["Market Total"] = (
    biggest_total_differences["Market Total"].round(1)
)

biggest_total_differences["Total Diff"] = (
    biggest_total_differences["Total Diff"].round(1)
)

# Biggest Changes from the Preseason Projection

This is one of the most important weekly diagnostics. It shows which games changed most because of new in season information.


In [50]:
biggest_projection_changes = (
    projection_board[
        [
            "Matchup",
            "Preseason Margin",
            "Weekly Margin",
            "Margin Change",
            "SU Pick",
            "Win Prob"
        ]
    ]
    .assign(
        Absolute_Change=lambda x: x["Margin Change"].abs()
    )
    .sort_values("Absolute_Change", ascending=False)
    .drop(columns="Absolute_Change")
)

display(biggest_projection_changes.round(3))


,Matchup,Preseason Margin,Weekly Margin,Margin Change,SU Pick,Win Prob
5,BAL @ HOU,2.043,11.344,9.302,HOU,0.788
2,PHI @ DAL,-1.075,7.466,8.541,DAL,0.700
13,WAS @ ARI,0.268,-7.891,-8.160,WAS,0.711
9,NYG @ IND,6.295,-1.305,-7.599,NYG,0.537
8,NO @ CIN,2.399,8.707,6.307,CIN,0.730
12,TEN @ JAX,10.224,15.366,5.142,JAX,0.860
10,NYJ @ MIA,5.254,0.584,-4.670,MIA,0.516
7,NE @ LAC,-0.503,-4.694,-4.191,NE,0.630
4,ATL @ MIN,2.303,6.194,3.891,MIN,0.669
11,SEA @ SF,-1.500,1.737,3.237,SF,0.549


In [51]:
biggest_projection_changes["Preseason Margin"] = (
    biggest_projection_changes["Preseason Margin"].round(2)
)

biggest_projection_changes["Weekly Margin"] = (
    biggest_projection_changes["Weekly Margin"].round(2)
)

biggest_projection_changes["Margin Change"] = (
    biggest_projection_changes["Margin Change"].round(2)
)

biggest_projection_changes["Win Prob"] = (
    biggest_projection_changes["Win Prob"].round(3)
)

# Sanity Checks


In [52]:
assert len(games) == len(target_schedule)
assert games["weekly_neutral_strength_diff"].notna().all()
assert games["expected_home_margin"].notna().all()
assert games["predicted_win_probability"].between(0.5, 1.0).all()
assert games["projected_total"].notna().all()

score_margin_check = (
    games["projected_home_score"]
    - games["projected_away_score"]
)

score_total_check = (
    games["projected_home_score"]
    + games["projected_away_score"]
)

assert np.allclose(
    score_margin_check,
    games["expected_home_margin"]
)

assert np.allclose(
    score_total_check,
    games["projected_total"]
)

print("All weekly game prediction sanity checks passed.")


All weekly game prediction sanity checks passed.


# Save Week Outputs

Everything is written only to the weekly layer.

The detailed parquet preserves all modeling fields for later evaluation. The CSV files provide the clean weekly projection board and largest model versus market differences.


In [53]:

detailed_path = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_projections.parquet"
)

current_games = games.copy()

if detailed_path.exists():
    previous_games = pd.read_parquet(detailed_path)

    current_game_ids = set(current_games["game_id"])

    frozen_games = previous_games[
        ~previous_games["game_id"].isin(current_game_ids)
    ].copy()

    detailed_save = pd.concat(
        [frozen_games, current_games],
        ignore_index=True
    )
else:
    detailed_save = current_games.copy()

detailed_save.to_parquet(
    detailed_path,
    index=False
)

# Build the complete projection board
projection_save = projection_board.copy()

projection_save["ATS Units"] = games["ats_units"].values
projection_save["Total Units"] = games["total_units"].values

rounding = {
    "Win Prob": 3,
    "ATS Diff": 2,
    "ATS Units": 1,
    "Model Total": 1,
    "Market Total": 1,
    "Total Diff": 1,
    "Total Units": 1,
    "Preseason Margin": 2,
    "Weekly Margin": 2,
    "Margin Change": 2
}

for column, decimals in rounding.items():
    projection_save[column] = projection_save[column].round(decimals)

projection_csv_path = (
    WEEK_OUTPUT_DIR
    / f"week_{TARGET_WEEK:02d}_projections.csv"
)

# Preserve previously saved games when refreshing projections
if projection_csv_path.exists():
    previous_projections = pd.read_csv(projection_csv_path)

    current_matchups = set(projection_save["Matchup"])

    frozen_projections = previous_projections[
        ~previous_projections["Matchup"].isin(current_matchups)
    ].copy()

    projection_save = pd.concat(
        [frozen_projections, projection_save],
        ignore_index=True
    )

assert projection_save["Matchup"].is_unique, (
    "Duplicate matchups found in saved projections."
)

projection_save.to_csv(
    projection_csv_path,
    index=False
)

# Save model-versus-market analysis
biggest_ats_differences.to_csv(
    WEEK_OUTPUT_DIR
    / f"week_{TARGET_WEEK:02d}_biggest_ats_differences.csv",
    index=False
)

biggest_total_differences.to_csv(
    WEEK_OUTPUT_DIR
    / f"week_{TARGET_WEEK:02d}_biggest_total_differences.csv",
    index=False
)

biggest_projection_changes.to_csv(
    WEEK_OUTPUT_DIR
    / f"week_{TARGET_WEEK:02d}_biggest_projection_changes.csv",
    index=False
)

print("Saved:")
print(detailed_path)
print(projection_csv_path)

print(
    f"\nTotal saved Week {TARGET_WEEK} projections: "
    f"{len(projection_save)}"
)

print(
    WEEK_OUTPUT_DIR
    / f"week_{TARGET_WEEK:02d}_biggest_ats_differences.csv"
)

print(
    WEEK_OUTPUT_DIR
    / f"week_{TARGET_WEEK:02d}_biggest_total_differences.csv"
)

print(
    WEEK_OUTPUT_DIR
    / f"week_{TARGET_WEEK:02d}_biggest_projection_changes.csv"
)


Saved:
..\..\data\processed\weekly\week_12_projections.parquet
..\..\weekly_projections\outputs\week_12\week_12_projections.csv

Total saved Week 12 projections: 16
..\..\weekly_projections\outputs\week_12\week_12_biggest_ats_differences.csv
..\..\weekly_projections\outputs\week_12\week_12_biggest_total_differences.csv
..\..\weekly_projections\outputs\week_12\week_12_biggest_projection_changes.csv


# Weekly Pipeline

For each future week, the intended order is:

`02_Weekly_Data_Update`  
→ `03_Inseason_Feature_Engineering`  
→ `04_Weekly_Team_Strength`  
→ `06_Week2_Projections`

The original preseason notebooks remain frozen throughout the season.

The next notebook will be `06_Weekly_Evaluation.ipynb`, which will compare locked weekly predictions against actual results after each week is complete.


In [54]:
ros_scoring_path = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_ros_scoring_ratings.parquet"
)

scoring_model[
    ["team", "current_off_ppg", "current_def_ppg"]
].to_parquet(
    ros_scoring_path,
    index=False
)

print("Saved:", ros_scoring_path)

Saved: ..\..\data\processed\weekly\week_12_ros_scoring_ratings.parquet
